# DAG Parameterization – Year Sensitivity Analysis
Evaluates whether model performance is stable across the three cohort years in the test set (2020, 2021, 2022).
The test set covers patients admitted on or after 2020; temporal stability suggests the model generalises without year-specific drift.

In [1]:
# Data manipulation and plotting
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import re
import glob

# Metrics
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, balanced_accuracy_score
from sklearn.calibration import calibration_curve
from pycalib.metrics import ECE

import xgboost

## Data preparation

In [2]:
# Load training data
X_train = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_imp.csv', index_col=0)
X_test  = pd.read_csv('../../Data Pre-processing/X_test_c12_w48_imp.csv',  index_col=0)

print(X_train.shape, X_test.shape)

# Remove features with near-zero variance
X_train = X_train.loc[:, X_train.var() >= 0.01]
X_test  = X_test.filter(items=X_train.columns)
print(X_train.shape, X_test.shape)

# MICE-imputed training set (used for DAG feature selection)
X_train_imp = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_mice.csv', index_col=0)

y_train = pd.read_csv('../../Data Pre-processing/y_train_c12_w48_imp.csv', index_col=[0,1]).groupby('uid').max()
y_test  = pd.read_csv('../../Data Pre-processing/y_test_c12_w48_imp.csv',  index_col=[0,1]).groupby('uid').max()

# Correlation-based feature selection (threshold used in original experiments)
correlation_threshold = 0.3
X_train_corr = X_train_imp.loc[:, X_train_imp.corrwith(y_train['Outcome']).abs() >= correlation_threshold]
print('Correlation-filtered features:', X_train_corr.shape)

(13054, 990) (3895, 990)
(13054, 811) (3895, 811)


Correlation-filtered features: (13054, 113)


## Year-split test sets
Join admission year from `patients.parquet` to the test UIDs and partition into 2020 / 2021 / 2022 subsets.

In [3]:
# Load admission-year metadata
patients = pd.read_parquet('../../Data Pre-processing/Preprocessing/data/real_cohort/patients.parquet')
uid_year = patients.set_index('uid')['arrive_yr']

# Map years onto test UIDs — UIDs absent from patients.parquet get NaN
test_years = uid_year.reindex(X_test.index)

n_missing = int(test_years.isna().sum())
print(f'Test UIDs without year info: {n_missing} (included in "All" evaluation only)')

# Create per-year subsets
TEST_YEARS = [2020, 2021, 2022]
X_test_by_year = {yr: X_test[test_years == yr] for yr in TEST_YEARS}
y_test_by_year = {yr: y_test[test_years == yr] for yr in TEST_YEARS}

print('\nTest set year distribution:')
for yr in TEST_YEARS:
    n     = len(X_test_by_year[yr])
    n_pos = int(y_test_by_year[yr]['Outcome'].sum())
    print(f'  {yr}: {n:5d} patients, {n_pos:3d} positive outcomes ({n_pos/n*100:.1f}%)')
n_all = len(X_test)
n_pos_all = int(y_test.Outcome.sum())
print(f'  All : {n_all:5d} patients, {n_pos_all:3d} positive outcomes ({n_pos_all/n_all*100:.1f}%)')

Test UIDs without year info: 154 (included in "All" evaluation only)

Test set year distribution:
  2020:  1060 patients, 148 positive outcomes (14.0%)
  2021:  1332 patients, 139 positive outcomes (10.4%)
  2022:  1349 patients, 122 positive outcomes (9.0%)
  All :  3895 patients, 431 positive outcomes (11.1%)


## Loading DAGs

In [4]:
# Load all adjacency files
adjacency_files = glob.glob("../DAGs/*_adjacency.csv")

dags = {}
dags['Control']     = nx.from_edgelist([(col, 'Outcome') for col in X_train_imp.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())
dags['Correlation'] = nx.from_edgelist([(col, 'Outcome') for col in X_train_corr.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())
for file in adjacency_files:
    dag_name = re.search(r'../DAGs/(.*)_adjacency.csv', file).group(1)
    dag_name = dag_name.replace('x', '$\\cap$')
    dag_name = dag_name.replace(' + ', ' $\\cup$ ')
    dags[dag_name] = nx.DiGraph(pd.read_csv(file, index_col=0))

dags.pop('Golem $\\cap$ PCMB')  # Remove problematic DAG (0 nodes associated with Outcome)
print('DAGs loaded:', list(dags.keys()))

DAGs loaded: ['Control', 'Correlation', 'Clinician Consensus $\\cup$ Golem', 'Simplified Clinician Consensus $\\cup$ Simplified PCMB', 'Simplified Clinician Consensus $\\cup$ Simplified Golem', 'Clinician Consensus $\\cup$ PCMB', 'Simplified Golem $\\cup$ Simplified PCMB', 'Clinician Consensus', 'Simplified Clinician Consensus', 'Clinician Consensus $\\cap$ PCMB', 'Golem', 'PCMB', 'Simplified Clinician Consensus $\\cap$ Simplified PCMB', 'Simplified PCMB', 'Clinician Consensus $\\cap$ Golem', 'Simplified Golem', 'Golem $\\cup$ PCMB', 'Simplified Golem $\\cap$ Simplified PCMB', 'Simplified Clinician Consensus $\\cap$ Simplified Golem']


## Helper functions

In [5]:
from MLstatkit import Bootstrapping

def performance_report(X, y, model):
    y_prob = model.predict_proba(X)[:, 1]

    n_bootstraps = 1000
    ap,        ap_lower,        ap_upper        = Bootstrapping(y.values, y_prob, metric_str='average_precision', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    auroc,     auroc_lower,     auroc_upper     = Bootstrapping(y.values, y_prob, metric_str='roc_auc',           n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    precision, precision_lower, precision_upper = Bootstrapping(y.values, y_prob, metric_str='precision',         n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    recall,    recall_lower,    recall_upper    = Bootstrapping(y.values, y_prob, metric_str='recall',            n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    f1,        f1_lower,        f1_upper        = Bootstrapping(y.values, y_prob, metric_str='f1',                n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    accuracy,  accuracy_lower,  accuracy_upper  = Bootstrapping(y.values, y_prob, metric_str='accuracy',          n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    ece = ECE(y.values, y_prob.reshape(-1, 1), bins=50)

    return {
        'Average Precision': f"{ap:.2f} ({ap_lower:.2f}, {ap_upper:.2f})",
        'AUROC':             f"{auroc:.2f} ({auroc_lower:.2f}, {auroc_upper:.2f})",
        'Precision':         f"{precision:.2f} ({precision_lower:.2f}, {precision_upper:.2f})",
        'Recall':            f"{recall:.2f} ({recall_lower:.2f}, {recall_upper:.2f})",
        'F1 Score':          f"{f1:.2f} ({f1_lower:.2f}, {f1_upper:.2f})",
        'Accuracy':          f"{accuracy:.2f} ({accuracy_lower:.2f}, {accuracy_upper:.2f})",
        'ECE':               f"{ece:.3f}",
    }

## Training function with year-stratified evaluation
Trains models on the full training set (identical to Experiment 1 in the main notebook) and then evaluates each trained model on:
- The full test set (`Year = 'All'`)
- Each year-split subset (`Year = 2020 / 2021 / 2022`)

In [6]:
def train_models_year_sensitivity(remove_drugs=False, remove_interventions=False):
    results = []
    preds_by_dag = {}  # {dag_name: {year_label: (y_true, y_prob)}}

    drugs = [
        'Midazolam', 'Fentanyl', 'Olanzapine', 'Haloperidol',
        'Dexmedetomidine', 'Lorazepam', 'Morphine', 'Hydromorphone',
        'Dopamine', 'Cisatracurium', 'Epinephrine', 'Norepinephrine',
        'Milrinone', 'Dobutamine',
    ]
    intervention_nodes = [
        'CRRT Therapy Type', 'ECMO Type', 'Ventilated',
        'Pupillary Reaction', 'Peds Coma Score', 'Endotracheal tube',
    ]

    for dag_name, dag in dags.items():
        if 'Simplified' not in dag_name and dag_name not in ['Control', 'Correlation']:
            continue

        print(f"Processing {dag_name} ({dag.number_of_nodes()} nodes, {dag.number_of_edges()} edges)")

        nodes_in_dag = list(dag.nodes())

        if 'Simplified' in dag_name or dag_name in ['Control', 'Correlation']:
            if remove_drugs:
                nodes_in_dag = [n for n in nodes_in_dag if n not in drugs]
            if remove_interventions:
                nodes_in_dag = [n for n in nodes_in_dag if n not in intervention_nodes]

            if dag_name == 'Correlation':
                features_in_dag = [col for col in X_train_corr.columns
                                   if any(re.search(rf'^{node}(_.+)?$', col) for node in nodes_in_dag)]
            else:
                features_in_dag = [col for col in X_train_imp.columns
                                   if any(re.search(rf'^{node}(_.+)?$', col) for node in nodes_in_dag)]

            n_biomarkers = len(nodes_in_dag) - 1

        else:
            if remove_drugs:
                nodes_in_dag = [n for n in nodes_in_dag if not any(d in n for d in drugs)]
            if remove_interventions:
                nodes_in_dag = [n for n in nodes_in_dag if not any(iv in n for iv in intervention_nodes)]

            features_in_dag = X_train_imp.filter(items=nodes_in_dag).columns.tolist()
            n_biomarkers = (X_train_imp.filter(items=nodes_in_dag)
                            .columns.str.replace(r'(_.+)?$', '', regex=True).nunique() - 1)

        # Train XGBoost
        xgb = xgboost.XGBClassifier(objective='binary:logistic', random_state=42, eval_metric='aucpr')
        xgb.fit(X_train.filter(features_in_dag), y_train.Outcome)

        # Evaluate on full test set and each year subset; store raw predictions for permutation tests
        eval_sets = [('All', X_test.filter(features_in_dag), y_test.Outcome)]
        for yr in TEST_YEARS:
            eval_sets.append((yr, X_test_by_year[yr].filter(features_in_dag), y_test_by_year[yr].Outcome))

        preds_by_dag[dag_name] = {}
        for year_label, X_eval, y_eval in eval_sets:
            y_prob = xgb.predict_proba(X_eval)[:, 1]
            preds_by_dag[dag_name][year_label] = (y_eval.values, y_prob)

            row = {'Model': 'XGB', 'DAG': dag_name, 'Year': year_label,
                   '# Features': len(features_in_dag), '# Biomarkers': n_biomarkers}
            row.update(performance_report(X_eval, y_eval, xgb))
            results.append(row)

    return pd.DataFrame(results), preds_by_dag

## Run year sensitivity experiment
Mirrors Experiment 1 from the main notebook (no drugs or interventions removed) with year-stratified evaluation.

In [7]:
results_df, preds_by_dag = train_models_year_sensitivity(remove_drugs=False, remove_interventions=False)

Processing Control (46 nodes, 45 edges)


Bootstrapping average_precision:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping average_precision:  13%|█▎        | 129/1000 [00:00<00:00, 1283.82it/s]

Bootstrapping average_precision:  26%|██▌       | 258/1000 [00:00<00:00, 1285.58it/s]

Bootstrapping average_precision:  39%|███▊      | 387/1000 [00:00<00:00, 1277.86it/s]

...

Bootstrapping f1: 100%|██████████| 1000/1000 [00:00<00:00, 1410.34it/s]

Bootstrapping accuracy:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping accuracy:  42%|████▏     | 415/1000 [00:00<00:00, 4143.48it/s]

Bootstrapping accuracy:  83%|████████▎ | 830/1000 [00:00<00:00, 4138.71it/s]

Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4134.77it/s]

In [8]:
results_df.to_csv('biomarker_counts_fixed/Year Sensitivity.csv', index=False)
results_df

,Model,DAG,Year,# Features,# Biomarkers,Average Precision,AUROC,Precision,Recall,F1 Score,Accuracy,ECE
0,XGB,Control,All,811,45,"0.81 (0.78, 0.84)","0.93 (0.91, 0.95)","0.94 (0.92, 0.95)","0.80 (0.78, 0.83)","0.85 (0.83, 0.87)","0.95 (0.94, 0.96)",0.089
1,XGB,Control,2020,811,45,"0.81 (0.76, 0.86)","0.93 (0.90, 0.95)","0.93 (0.90, 0.96)","0.82 (0.78, 0.85)","0.86 (0.83, 0.89)","0.94 (0.93, 0.96)",0.110
2,XGB,Control,2021,811,45,"0.82 (0.77, 0.88)","0.95 (0.93, 0.97)","0.95 (0.93, 0.98)","0.79 (0.74, 0.83)","0.85 (0.81, 0.88)","0.95 (0.94, 0.96)",0.079
3,XGB,Control,2022,811,45,"0.84 (0.78, 0.89)","0.93 (0.90, 0.96)","0.94 (0.91, 0.97)","0.84 (0.80, 0.88)","0.88 (0.85, 0.91)","0.97 (0.96, 0.97)",0.083
4,XGB,Correlation,All,113,37,"0.75 (0.71, 0.78)","0.90 (0.88, 0.91)","0.90 (0.88, 0.92)","0.78 (0.76, 0.81)","0.83 (0.81, 0.85)","0.94 (0.94, 0.95)",0.108
5,XGB,Correlation,2020,113,37,"0.75 (0.69, 0.81)","0.89 (0.86, 0.92)","0.90 (0.86, 0.93)","0.77 (0.73, 0.81)","0.82 (0.78, 0.85)","0.93 (0.91, 0.94)",0.125
6,XGB,Correlation,2021,113,37,"0.77 (0.70, 0.82)","0.91 (0.89, 0.94)","0.90 (0.86, 0.94)","0.80 (0.76, 0.83)","0.84 (0.80, 0.87)","0.95 (0.94, 0.96)",0.104
7,XGB,Correlation,2022,113,37,"0.77 (0.70, 0.83)","0.89 (0.85, 0.93)","0.90 (0.87, 0.94)","0.81 (0.77, 0.85)","0.85 (0.82, 0.88)","0.96 (0.95, 0.97)",0.099
8,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,All,334,19,"0.77 (0.74, 0.81)","0.91 (0.89, 0.93)","0.92 (0.91, 0.94)","0.79 (0.77, 0.82)","0.84 (0.82, 0.87)","0.95 (0.94, 0.96)",0.092
9,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,2020,334,19,"0.77 (0.71, 0.82)","0.90 (0.87, 0.93)","0.92 (0.89, 0.95)","0.78 (0.74, 0.82)","0.83 (0.79, 0.86)","0.93 (0.92, 0.95)",0.112


## Summary: AUROC, AUPRC, and Normalized AUPRC by Year
AUPRC depends on class prevalence (π), which differs across years (2020: 14%, 2021: 10%, 2022: 9%).
The **skill score** `(AUPRC − π) / (1 − π)` re-anchors to [0 = random, 1 = perfect] and makes years comparable.
CI bounds transform linearly since π is fixed per year.

In [9]:
def extract_point_ci(s):
    """Parse 'value (lower, upper)' string into three floats."""
    try:
        s = str(s).replace('(', '').replace(')', '').replace(',', '')
        parts = s.split()
        return float(parts[0]), float(parts[1]), float(parts[2])
    except Exception:
        v = float(s)
        return v, v, v

def extract_point(s):
    return extract_point_ci(s)[0]

def normalize_auprc(auprc_str, pi):
    """Apply skill-score normalization to an AUPRC string with CI."""
    pt, lo, hi = extract_point_ci(auprc_str)
    norm    = (pt - pi) / (1 - pi)
    norm_lo = (lo - pi) / (1 - pi)
    norm_hi = (hi - pi) / (1 - pi)
    return f"{norm:.2f} ({norm_lo:.2f}, {norm_hi:.2f})"

# Per-year prevalence (π) used for normalization
prevalence = {yr: float(y_test_by_year[yr]['Outcome'].mean()) for yr in TEST_YEARS}
prevalence['All'] = float(y_test['Outcome'].mean())
print('Prevalence by year:', {k: f"{v:.3f}" for k, v in prevalence.items()})

# Add normalized AUPRC column
summary = results_df.copy()
summary['Normalized AUPRC'] = summary.apply(
    lambda row: normalize_auprc(row['Average Precision'], prevalence[row['Year']]),
    axis=1
)

# Numeric point estimates for plotting / delta tables
for col in ['Average Precision', 'AUROC', 'Normalized AUPRC']:
    summary[col + '_pt'] = summary[col].apply(extract_point)

# Pivot: rows = DAG, columns = Year
for metric_col, label in [
    ('Average Precision_pt', 'Average Precision (raw)'),
    ('Normalized AUPRC_pt',  'Normalized AUPRC (skill score)'),
    ('AUROC_pt',             'AUROC'),
]:
    pivot = summary.pivot_table(index='DAG', columns='Year', values=metric_col)
    pivot.columns = [str(c) for c in pivot.columns]
    col_order = ['All'] + [str(yr) for yr in TEST_YEARS]
    pivot = pivot[[c for c in col_order if c in pivot.columns]]
    print(f'\n=== {label} ===')
    display(pivot.round(3))

Prevalence by year: {2020: '0.140', 2021: '0.104', 2022: '0.090', 'All': '0.111'}

=== Average Precision (raw) ===


,All,2020,2021,2022
DAG,,,,
Control,0.81,0.81,0.82,0.84
Correlation,0.75,0.75,0.77,0.77
Simplified Clinician Consensus,0.78,0.80,0.78,0.81
Simplified Clinician Consensus $\cap$ Simplified Golem,0.74,0.78,0.73,0.74
Simplified Clinician Consensus $\cap$ Simplified PCMB,0.79,0.80,0.79,0.81
Simplified Clinician Consensus $\cup$ Simplified Golem,0.79,0.81,0.79,0.82
Simplified Clinician Consensus $\cup$ Simplified PCMB,0.77,0.77,0.81,0.80
Simplified Golem,0.77,0.78,0.79,0.80
Simplified Golem $\cap$ Simplified PCMB,0.76,0.78,0.77,0.78



=== Normalized AUPRC (skill score) ===


,All,2020,2021,2022
DAG,,,,
Control,0.79,0.78,0.80,0.82
Correlation,0.72,0.71,0.74,0.75
Simplified Clinician Consensus,0.75,0.77,0.75,0.79
Simplified Clinician Consensus $\cap$ Simplified Golem,0.71,0.74,0.70,0.71
Simplified Clinician Consensus $\cap$ Simplified PCMB,0.76,0.77,0.77,0.79
Simplified Clinician Consensus $\cup$ Simplified Golem,0.76,0.78,0.77,0.80
Simplified Clinician Consensus $\cup$ Simplified PCMB,0.74,0.73,0.79,0.78
Simplified Golem,0.74,0.74,0.77,0.78
Simplified Golem $\cap$ Simplified PCMB,0.73,0.74,0.74,0.76



=== AUROC ===


,All,2020,2021,2022
DAG,,,,
Control,0.93,0.93,0.95,0.93
Correlation,0.90,0.89,0.91,0.89
Simplified Clinician Consensus,0.91,0.91,0.92,0.91
Simplified Clinician Consensus $\cap$ Simplified Golem,0.89,0.90,0.89,0.88
Simplified Clinician Consensus $\cap$ Simplified PCMB,0.91,0.91,0.92,0.92
Simplified Clinician Consensus $\cup$ Simplified Golem,0.91,0.92,0.91,0.92
Simplified Clinician Consensus $\cup$ Simplified PCMB,0.91,0.90,0.93,0.91
Simplified Golem,0.91,0.91,0.93,0.92
Simplified Golem $\cap$ Simplified PCMB,0.90,0.91,0.91,0.91


In [10]:
cols_to_save = ['DAG', 'Year', 'Average Precision', 'Normalized AUPRC', 'AUROC',
                'Precision', 'Recall', 'F1 Score', 'Accuracy', 'ECE',
                '# Features', '# Biomarkers']
summary[cols_to_save].to_csv('biomarker_counts_fixed/Year Sensitivity - Normalized.csv', index=False)

## Plot: AUROC and AUPRC by Year
Line plot across years for each DAG and model type, making any temporal drift immediately visible.

In [11]:
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'figure.dpi': 150,
    'font.size': 11,
    'lines.linewidth': 1.5,
    'figure.constrained_layout.use': True,
    'pdf.fonttype': 42,
})

year_only = summary[summary['Year'].isin(TEST_YEARS)].copy()
year_only['Year'] = year_only['Year'].astype(int)

metrics_to_plot = [
    ('AUROC_pt',             'AUROC'),
    ('Average Precision_pt', 'Average Precision (raw)'),
    ('Normalized AUPRC_pt',  'Normalized AUPRC (skill score)'),
]

fig, axes = plt.subplots(1, 3)
fig.suptitle('XGB – Performance by Test-Set Year')

for ax, (metric, ylabel) in zip(axes, metrics_to_plot):
    for dag_name, grp in year_only.groupby('DAG'):
        grp_sorted = grp.sort_values('Year')
        ax.plot(grp_sorted['Year'], grp_sorted[metric], marker='o', label=dag_name, alpha=0.7)
    ax.set_xlabel('Year')
    ax.set_ylabel(ylabel)
    ax.set_xticks(TEST_YEARS)
    ax.grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.3), fontsize=8, frameon=False)

plt.savefig('biomarker_counts_fixed/Year Sensitivity - XGB.pdf', bbox_inches='tight', dpi=300)
plt.show()
plt.rcParams.update(plt.rcParamsDefault)

/var/folders/17/ccxjtf7x73b7_wzzqdvhv3h80000gq/T/ipykernel_58532/3808486603.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Year-over-year delta table
Shows how much AUROC and Average Precision shift from 2020 → 2021 and 2021 → 2022 for each DAG × Model.

In [12]:
for metric_col, label in [
    ('AUROC_pt',             'AUROC'),
    ('Average Precision_pt', 'Average Precision (raw)'),
    ('Normalized AUPRC_pt',  'Normalized AUPRC (skill score)'),
]:
    pivot = summary.pivot_table(index='DAG', columns='Year', values=metric_col)
    pivot.columns = [str(c) for c in pivot.columns]
    delta = pd.DataFrame(index=pivot.index)
    delta['2020→2021'] = pivot['2021'] - pivot['2020']
    delta['2021→2022'] = pivot['2022'] - pivot['2021']
    delta['2020→2022'] = pivot['2022'] - pivot['2020']
    print(f'\n=== {label} – year-over-year delta ===')
    display(delta.round(3))


=== AUROC – year-over-year delta ===


,2020→2021,2021→2022,2020→2022
DAG,,,
Control,0.02,-0.02,0.00
Correlation,0.02,-0.02,0.00
Simplified Clinician Consensus,0.01,-0.01,0.00
Simplified Clinician Consensus $\cap$ Simplified Golem,-0.01,-0.01,-0.02
Simplified Clinician Consensus $\cap$ Simplified PCMB,0.01,0.00,0.01
Simplified Clinician Consensus $\cup$ Simplified Golem,-0.01,0.01,0.00
Simplified Clinician Consensus $\cup$ Simplified PCMB,0.03,-0.02,0.01
Simplified Golem,0.02,-0.01,0.01
Simplified Golem $\cap$ Simplified PCMB,0.00,0.00,0.00



=== Average Precision (raw) – year-over-year delta ===


,2020→2021,2021→2022,2020→2022
DAG,,,
Control,0.01,0.02,0.03
Correlation,0.02,0.00,0.02
Simplified Clinician Consensus,-0.02,0.03,0.01
Simplified Clinician Consensus $\cap$ Simplified Golem,-0.05,0.01,-0.04
Simplified Clinician Consensus $\cap$ Simplified PCMB,-0.01,0.02,0.01
Simplified Clinician Consensus $\cup$ Simplified Golem,-0.02,0.03,0.01
Simplified Clinician Consensus $\cup$ Simplified PCMB,0.04,-0.01,0.03
Simplified Golem,0.01,0.01,0.02
Simplified Golem $\cap$ Simplified PCMB,-0.01,0.01,0.00



=== Normalized AUPRC (skill score) – year-over-year delta ===


,2020→2021,2021→2022,2020→2022
DAG,,,
Control,0.02,0.02,0.04
Correlation,0.03,0.01,0.04
Simplified Clinician Consensus,-0.02,0.04,0.02
Simplified Clinician Consensus $\cap$ Simplified Golem,-0.04,0.01,-0.03
Simplified Clinician Consensus $\cap$ Simplified PCMB,0.00,0.02,0.02
Simplified Clinician Consensus $\cup$ Simplified Golem,-0.01,0.03,0.02
Simplified Clinician Consensus $\cup$ Simplified PCMB,0.06,-0.01,0.05
Simplified Golem,0.03,0.01,0.04
Simplified Golem $\cap$ Simplified PCMB,0.00,0.02,0.02


## Permutation test: pairwise year comparisons
Tests whether the same model performs significantly differently across years.

**Design:** pool predictions from years A and B, randomly shuffle year labels (preserving group sizes), recompute the metric difference, repeat 1000 times. The p-value is the fraction of permuted differences with |Δ| ≥ |observed Δ| (two-tailed).

Note: raw AUPRC and AUROC are used inside the permutation loop (not normalized AUPRC), because permuting year labels changes the within-permutation prevalence, making normalization ill-defined. Normalized AUPRC values from the table above provide the prevalence-adjusted magnitudes.

In [13]:
from sklearn.metrics import average_precision_score, roc_auc_score
from itertools import combinations

def permutation_test_across_years(y_true_A, y_prob_A, y_true_B, y_prob_B,
                                   metric_str='average_precision',
                                   n_permutations=1000, random_state=42):
    """
    Two-tailed permutation test comparing the same model on two year subsets.

    H0: the metric does not differ between year A and year B.
    Permutes year labels (pooling A+B) to build the null distribution of |Δ metric|.
    """
    metric_fn = {'average_precision': average_precision_score,
                 'roc_auc': roc_auc_score}[metric_str]

    obs_A = metric_fn(y_true_A, y_prob_A)
    obs_B = metric_fn(y_true_B, y_prob_B)
    obs_diff = abs(obs_A - obs_B)

    y_true_pool = np.concatenate([y_true_A, y_true_B])
    y_prob_pool = np.concatenate([y_prob_A, y_prob_B])
    n_A = len(y_true_A)

    rng = np.random.default_rng(random_state)
    null_diffs = []
    for _ in range(n_permutations):
        idx = rng.permutation(len(y_true_pool))
        try:
            diff = abs(
                metric_fn(y_true_pool[idx[:n_A]], y_prob_pool[idx[:n_A]])
                - metric_fn(y_true_pool[idx[n_A:]], y_prob_pool[idx[n_A:]])
            )
        except Exception:
            diff = 0.0
        null_diffs.append(diff)

    p_value = float(np.mean(np.array(null_diffs) >= obs_diff))
    return obs_A, obs_B, obs_diff, p_value


# Run pairwise tests for each DAG and metric
year_pairs = list(combinations(TEST_YEARS, 2))
perm_rows = []

for dag_name, year_preds in preds_by_dag.items():
    for yr_A, yr_B in year_pairs:
        y_true_A, y_prob_A = year_preds[yr_A]
        y_true_B, y_prob_B = year_preds[yr_B]

        for metric_str, metric_label in [('average_precision', 'AUPRC'),
                                          ('roc_auc',           'AUROC')]:
            obs_A, obs_B, obs_diff, p_value = permutation_test_across_years(
                y_true_A, y_prob_A, y_true_B, y_prob_B,
                metric_str=metric_str, n_permutations=1000, random_state=42,
            )
            perm_rows.append({
                'DAG':        dag_name,
                'Metric':     metric_label,
                'Year A':     yr_A,
                'Year B':     yr_B,
                f'{metric_label} (A)': round(obs_A, 3),
                f'{metric_label} (B)': round(obs_B, 3),
                '|Δ|':        round(obs_diff, 3),
                'p-value':    round(p_value, 3),
                'Significant (p<0.05)': p_value < 0.05,
            })

perm_df = pd.DataFrame(perm_rows)
perm_df.to_csv('biomarker_counts_fixed/Year Sensitivity - Permutation Tests.csv', index=False)
perm_df

,DAG,Metric,Year A,Year B,AUPRC (A),AUPRC (B),|Δ|,p-value,Significant (p<0.05),AUROC (A),AUROC (B)
0,Control,AUPRC,2020,2021,0.814,0.822,0.008,0.827,False,NaN,NaN
1,Control,AUROC,2020,2021,NaN,NaN,0.021,0.224,False,0.926,0.947
2,Control,AUPRC,2020,2022,0.814,0.835,0.021,0.592,False,NaN,NaN
3,Control,AUROC,2020,2022,NaN,NaN,0.007,0.721,False,0.926,0.932
4,Control,AUPRC,2021,2022,0.822,0.835,0.013,0.721,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
61,Simplified Clinician Consensus $\cap$ Simplifi...,AUROC,2020,2021,NaN,NaN,0.004,0.855,False,0.896,0.892
62,Simplified Clinician Consensus $\cap$ Simplifi...,AUPRC,2020,2022,0.783,0.743,0.041,0.370,False,NaN,NaN
63,Simplified Clinician Consensus $\cap$ Simplifi...,AUROC,2020,2022,NaN,NaN,0.012,0.659,False,0.896,0.884
64,Simplified Clinician Consensus $\cap$ Simplifi...,AUPRC,2021,2022,0.726,0.743,0.017,0.718,False,NaN,NaN


In [14]:
# Summary: which year pairs show significant differences, and for which DAGs?
print("=== Significant year differences (p < 0.05) ===\n")
for metric_label in ['AUPRC', 'AUROC']:
    sub = perm_df[perm_df['Metric'] == metric_label]
    sig = sub[sub['Significant (p<0.05)']]
    print(f"{metric_label}: {len(sig)} / {len(sub)} comparisons significant")
    if len(sig):
        display(sig[['DAG', 'Year A', 'Year B', '|Δ|', 'p-value']].reset_index(drop=True))
    print()

=== Significant year differences (p < 0.05) ===

AUPRC: 0 / 33 comparisons significant

AUROC: 0 / 33 comparisons significant



## Permutation test on AUCNPR (prevalence-normalized)
Repeats the pairwise year comparison on **AUCNPR** instead of raw AUPRC. Unlike the raw test, each year subset is normalized by its **own** prevalence anchor ($\pi$) before the difference is taken, so this is a genuinely distinct test rather than a monotone relabelling — the observed $|\Delta|$ and the permutation null both differ from the raw-AUPRC version.

This addresses the concern that the apparent year-over-year rise in raw AUPRC is largely a prevalence artifact (prevalence falls 14.0% → 10.4% → 9.0% across 2020–2022). Saved to `Year Sensitivity - Permutation AUCNPR.csv`.

In [15]:
# AUCNPR (prevalence-normalized AUPRC) permutation test.
# Each year subset keeps its OWN prevalence anchor throughout the permutation, so the
# null asks whether prevalence-adjusted discrimination differs between years.
#   AUCNPR   = (AUPRC - AUPR_min) / (1 - AUPR_min)
#   AUPR_min = 1 + (1 - pi) * ln(1 - pi) / pi          (Boyd, Eng & Page, ECML PKDD 2012)

def aucpr_min(pi):
    """Worst-case (minimum achievable) AUPRC at prevalence pi (Boyd et al. 2012)."""
    return 1 + (1 - pi) * np.log(1 - pi) / pi


def permutation_test_aucnpr_across_years(y_true_A, y_prob_A, pi_A,
                                         y_true_B, y_prob_B, pi_B,
                                         n_permutations=1000, random_state=42):
    """
    Two-tailed permutation test on AUCNPR comparing the same model on two year subsets.

    H0: prevalence-adjusted discrimination (AUCNPR) does not differ between year A and B.
    Each side is normalized by its original-year prevalence (pi_A, pi_B); year labels are
    then pooled and shuffled to build the null distribution of |Delta AUCNPR|.
    """
    min_A, min_B = aucpr_min(pi_A), aucpr_min(pi_B)
    npr_A = lambda yt, yp: (average_precision_score(yt, yp) - min_A) / (1 - min_A)
    npr_B = lambda yt, yp: (average_precision_score(yt, yp) - min_B) / (1 - min_B)

    obs_A, obs_B = npr_A(y_true_A, y_prob_A), npr_B(y_true_B, y_prob_B)
    obs_diff = abs(obs_A - obs_B)

    y_true_pool = np.concatenate([y_true_A, y_true_B])
    y_prob_pool = np.concatenate([y_prob_A, y_prob_B])
    n_A = len(y_true_A)

    rng = np.random.default_rng(random_state)
    null_diffs = []
    for _ in range(n_permutations):
        idx = rng.permutation(len(y_true_pool))
        try:
            diff = abs(
                npr_A(y_true_pool[idx[:n_A]], y_prob_pool[idx[:n_A]])
                - npr_B(y_true_pool[idx[n_A:]], y_prob_pool[idx[n_A:]])
            )
        except Exception:
            diff = 0.0
        null_diffs.append(diff)

    p_value = float(np.mean(np.array(null_diffs) >= obs_diff))
    return obs_A, obs_B, obs_diff, p_value


# Run AUCNPR pairwise tests for each DAG (reuses year_pairs from the raw-AUPRC test above)
perm_npr_rows = []
for dag_name, year_preds in preds_by_dag.items():
    for yr_A, yr_B in year_pairs:
        y_true_A, y_prob_A = year_preds[yr_A]
        y_true_B, y_prob_B = year_preds[yr_B]

        obs_A, obs_B, obs_diff, p_value = permutation_test_aucnpr_across_years(
            y_true_A, y_prob_A, prevalence[yr_A],
            y_true_B, y_prob_B, prevalence[yr_B],
            n_permutations=1000, random_state=42,
        )
        perm_npr_rows.append({
            'DAG':        dag_name,
            'Metric':     'AUCNPR',
            'Year A':     yr_A,
            'Year B':     yr_B,
            'AUCNPR (A)': round(obs_A, 3),
            'AUCNPR (B)': round(obs_B, 3),
            '|Δ|':        round(obs_diff, 3),
            'p-value':    round(p_value, 3),
            'Significant (p<0.05)': p_value < 0.05,
        })

perm_npr_df = pd.DataFrame(perm_npr_rows)
perm_npr_df.to_csv('biomarker_counts_fixed/Year Sensitivity - Permutation AUCNPR.csv', index=False)

n_sig = int(perm_npr_df['Significant (p<0.05)'].sum())
print(f"AUCNPR: {n_sig} / {len(perm_npr_df)} comparisons significant (p < 0.05)")
perm_npr_df

AUCNPR: 0 / 33 comparisons significant (p < 0.05)


,DAG,Metric,Year A,Year B,AUCNPR (A),AUCNPR (B),|Δ|,p-value,Significant (p<0.05)
0,Control,AUCNPR,2020,2021,0.800,0.812,0.013,0.756,False
1,Control,AUCNPR,2020,2022,0.800,0.827,0.028,0.501,False
2,Control,AUCNPR,2021,2022,0.812,0.827,0.015,0.697,False
3,Correlation,AUCNPR,2020,2021,0.733,0.753,0.020,0.675,False
4,Correlation,AUCNPR,2020,2022,0.733,0.754,0.021,0.684,False
5,Correlation,AUCNPR,2021,2022,0.753,0.754,0.001,0.985,False
6,Simplified Clinician Consensus $\cup$ Simplifi...,AUCNPR,2020,2021,0.751,0.795,0.045,0.334,False
7,Simplified Clinician Consensus $\cup$ Simplifi...,AUCNPR,2020,2022,0.751,0.789,0.038,0.433,False
8,Simplified Clinician Consensus $\cup$ Simplifi...,AUCNPR,2021,2022,0.795,0.789,0.006,0.861,False
9,Simplified Clinician Consensus $\cup$ Simplifi...,AUCNPR,2020,2021,0.795,0.778,0.017,0.688,False


# Experiment 2: Vitals & Labs Only (No Drugs or Interventions)
Re-runs the full year-sensitivity pipeline after removing all drug and intervention features, leaving only vitals and laboratory biomarkers (mirrors *Experiment 3: No Drugs or Interventions* in the main notebook). All helper functions, prevalence anchors ($\pi$), and year pairs defined above are reused.

**Note:** once drugs and interventions are removed, some simplified DAGs reduce to identical feature sets (e.g. SCC $\cup$ SPCMB $\equiv$ SPCMB at 131 features; SCC $\equiv$ SCC $\cap$ SPCMB at 93 features), so those rows carry identical metrics.

In [16]:
# Vitals & labs only: drop drug and intervention features
results_df_vl, preds_by_dag_vl = train_models_year_sensitivity(remove_drugs=True, remove_interventions=True)
results_df_vl.to_csv('biomarker_counts_fixed/Year Sensitivity - Vitals Labs.csv', index=False)
results_df_vl

Processing Control (46 nodes, 45 edges)


Bootstrapping average_precision:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping average_precision:  13%|█▎        | 133/1000 [00:00<00:00, 1328.99it/s]

Bootstrapping average_precision:  27%|██▋       | 266/1000 [00:00<00:00, 1321.26it/s]

Bootstrapping average_precision:  40%|████      | 400/1000 [00:00<00:00, 1328.65it/s]

...

Bootstrapping accuracy:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping accuracy:  42%|████▏     | 420/1000 [00:00<00:00, 4193.84it/s]

Bootstrapping accuracy:  84%|████████▍ | 840/1000 [00:00<00:00, 4195.74it/s]

Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 4189.00it/s]

,Model,DAG,Year,# Features,# Biomarkers,Average Precision,AUROC,Precision,Recall,F1 Score,Accuracy,ECE
0,XGB,Control,All,521,28,"0.78 (0.75, 0.82)","0.92 (0.90, 0.93)","0.90 (0.88, 0.92)","0.80 (0.78, 0.82)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.093
1,XGB,Control,2020,521,28,"0.79 (0.73, 0.85)","0.91 (0.88, 0.94)","0.91 (0.88, 0.94)","0.80 (0.76, 0.84)","0.84 (0.81, 0.87)","0.93 (0.92, 0.95)",0.113
2,XGB,Control,2021,521,28,"0.80 (0.74, 0.85)","0.93 (0.91, 0.96)","0.91 (0.88, 0.95)","0.80 (0.77, 0.84)","0.85 (0.81, 0.88)","0.95 (0.94, 0.96)",0.087
3,XGB,Control,2022,521,28,"0.80 (0.74, 0.86)","0.93 (0.89, 0.96)","0.90 (0.86, 0.94)","0.82 (0.78, 0.86)","0.86 (0.82, 0.89)","0.96 (0.95, 0.97)",0.086
4,XGB,Correlation,All,53,20,"0.75 (0.71, 0.78)","0.89 (0.87, 0.91)","0.90 (0.88, 0.92)","0.78 (0.76, 0.80)","0.83 (0.80, 0.85)","0.94 (0.93, 0.95)",0.114
5,XGB,Correlation,2020,53,20,"0.75 (0.69, 0.81)","0.89 (0.85, 0.92)","0.88 (0.84, 0.91)","0.76 (0.72, 0.80)","0.80 (0.76, 0.84)","0.92 (0.90, 0.93)",0.131
6,XGB,Correlation,2021,53,20,"0.76 (0.69, 0.82)","0.89 (0.86, 0.93)","0.92 (0.89, 0.95)","0.80 (0.76, 0.84)","0.85 (0.81, 0.88)","0.95 (0.94, 0.96)",0.109
7,XGB,Correlation,2022,53,20,"0.78 (0.72, 0.84)","0.91 (0.87, 0.94)","0.93 (0.90, 0.96)","0.82 (0.78, 0.86)","0.87 (0.83, 0.90)","0.96 (0.95, 0.97)",0.106
8,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,All,150,8,"0.76 (0.73, 0.80)","0.91 (0.89, 0.93)","0.90 (0.88, 0.92)","0.79 (0.76, 0.81)","0.83 (0.81, 0.85)","0.94 (0.94, 0.95)",0.104
9,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,2020,150,8,"0.78 (0.72, 0.83)","0.90 (0.87, 0.93)","0.92 (0.89, 0.95)","0.80 (0.75, 0.83)","0.84 (0.80, 0.87)","0.93 (0.92, 0.95)",0.121


In [17]:
# Skill-score and AUCNPR point estimates for the vitals & labs model (reuses helpers above)
summary_vl = results_df_vl.copy()
summary_vl['Normalized AUPRC'] = summary_vl.apply(
    lambda r: normalize_auprc(r['Average Precision'], prevalence[r['Year']]), axis=1)

def aucnpr_point(auprc_str, pi):
    """AUCNPR point estimate from an AUPRC CI string (Boyd et al. 2012)."""
    return (extract_point(auprc_str) - aucpr_min(pi)) / (1 - aucpr_min(pi))

summary_vl['AUCNPR']               = summary_vl.apply(lambda r: aucnpr_point(r['Average Precision'], prevalence[r['Year']]), axis=1)
summary_vl['Average Precision_pt'] = summary_vl['Average Precision'].apply(extract_point)
summary_vl['AUROC_pt']             = summary_vl['AUROC'].apply(extract_point)

cols_to_save = ['DAG', 'Year', 'Average Precision', 'Normalized AUPRC', 'AUROC',
                'Precision', 'Recall', 'F1 Score', 'Accuracy', 'ECE', '# Features', '# Biomarkers']
summary_vl[cols_to_save].to_csv('biomarker_counts_fixed/Year Sensitivity - Vitals Labs - Normalized.csv', index=False)

# Table 1: Raw AUPRC + AUCNPR, wide by year, ordered by AUCNPR on the full test set
raw_piv = summary_vl.pivot_table(index='DAG', columns='Year', values='Average Precision_pt')
npr_piv = summary_vl.pivot_table(index='DAG', columns='Year', values='AUCNPR')
raw_piv.columns = [f'AUPRC_{c}'  for c in raw_piv.columns]
npr_piv.columns = [f'AUCNPR_{c}' for c in npr_piv.columns]
table1_vl = pd.concat([raw_piv, npr_piv], axis=1).sort_values('AUCNPR_All', ascending=False)
table1_vl.to_csv('biomarker_counts_fixed/Year Sensitivity - Vitals Labs - Table1 AUPRC AUCNPR.csv')
table1_vl.round(3)

,AUPRC_2020,AUPRC_2021,AUPRC_2022,AUPRC_All,AUCNPR_2020,AUCNPR_2021,AUCNPR_2022,AUCNPR_All
DAG,,,,,,,,
Control,0.79,0.80,0.80,0.78,0.773,0.789,0.790,0.767
Simplified Golem $\cup$ Simplified PCMB,0.77,0.79,0.79,0.77,0.752,0.778,0.780,0.756
Simplified Clinician Consensus $\cup$ Simplified Golem,0.76,0.79,0.79,0.76,0.741,0.778,0.780,0.745
Simplified Clinician Consensus $\cup$ Simplified PCMB,0.78,0.77,0.78,0.76,0.763,0.757,0.769,0.745
Simplified Golem,0.74,0.79,0.81,0.76,0.719,0.778,0.801,0.745
Simplified PCMB,0.76,0.79,0.80,0.76,0.741,0.778,0.790,0.745
Correlation,0.75,0.76,0.78,0.75,0.730,0.746,0.769,0.735
Simplified Clinician Consensus,0.75,0.74,0.76,0.73,0.730,0.725,0.748,0.714
Simplified Clinician Consensus $\cap$ Simplified PCMB,0.75,0.71,0.77,0.73,0.730,0.693,0.759,0.714


In [18]:
# Raw AUPRC / AUROC permutation test — vitals & labs model
perm_rows_vl = []
for dag_name, year_preds in preds_by_dag_vl.items():
    for yr_A, yr_B in year_pairs:
        y_true_A, y_prob_A = year_preds[yr_A]
        y_true_B, y_prob_B = year_preds[yr_B]

        for metric_str, metric_label in [('average_precision', 'AUPRC'),
                                          ('roc_auc',           'AUROC')]:
            obs_A, obs_B, obs_diff, p_value = permutation_test_across_years(
                y_true_A, y_prob_A, y_true_B, y_prob_B,
                metric_str=metric_str, n_permutations=1000, random_state=42,
            )
            perm_rows_vl.append({
                'DAG':        dag_name,
                'Metric':     metric_label,
                'Year A':     yr_A,
                'Year B':     yr_B,
                f'{metric_label} (A)': round(obs_A, 3),
                f'{metric_label} (B)': round(obs_B, 3),
                '|Δ|':        round(obs_diff, 3),
                'p-value':    round(p_value, 3),
                'Significant (p<0.05)': p_value < 0.05,
            })

perm_df_vl = pd.DataFrame(perm_rows_vl)
perm_df_vl.to_csv('biomarker_counts_fixed/Year Sensitivity - Vitals Labs - Permutation Tests.csv', index=False)
for metric_label in ['AUPRC', 'AUROC']:
    sub = perm_df_vl[perm_df_vl['Metric'] == metric_label]
    print(f"{metric_label}: {int(sub['Significant (p<0.05)'].sum())} / {len(sub)} comparisons significant (p < 0.05)")
perm_df_vl

AUPRC: 0 / 33 comparisons significant (p < 0.05)
AUROC: 1 / 33 comparisons significant (p < 0.05)


,DAG,Metric,Year A,Year B,AUPRC (A),AUPRC (B),|Δ|,p-value,Significant (p<0.05),AUROC (A),AUROC (B)
0,Control,AUPRC,2020,2021,0.794,0.798,0.004,0.925,False,NaN,NaN
1,Control,AUROC,2020,2021,NaN,NaN,0.026,0.214,False,0.908,0.934
2,Control,AUPRC,2020,2022,0.794,0.801,0.007,0.854,False,NaN,NaN
3,Control,AUROC,2020,2022,NaN,NaN,0.018,0.391,False,0.908,0.927
4,Control,AUPRC,2021,2022,0.798,0.801,0.003,0.941,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
61,Simplified Clinician Consensus $\cap$ Simplifi...,AUROC,2020,2021,NaN,NaN,0.006,0.829,False,0.868,0.862
62,Simplified Clinician Consensus $\cap$ Simplifi...,AUPRC,2020,2022,0.696,0.662,0.034,0.523,False,NaN,NaN
63,Simplified Clinician Consensus $\cap$ Simplifi...,AUROC,2020,2022,NaN,NaN,0.009,0.750,False,0.868,0.877
64,Simplified Clinician Consensus $\cap$ Simplifi...,AUPRC,2021,2022,0.665,0.662,0.003,0.953,False,NaN,NaN


In [19]:
# AUCNPR (prevalence-normalized) permutation test — vitals & labs model
perm_npr_rows_vl = []
for dag_name, year_preds in preds_by_dag_vl.items():
    for yr_A, yr_B in year_pairs:
        y_true_A, y_prob_A = year_preds[yr_A]
        y_true_B, y_prob_B = year_preds[yr_B]

        obs_A, obs_B, obs_diff, p_value = permutation_test_aucnpr_across_years(
            y_true_A, y_prob_A, prevalence[yr_A],
            y_true_B, y_prob_B, prevalence[yr_B],
            n_permutations=1000, random_state=42,
        )
        perm_npr_rows_vl.append({
            'DAG':        dag_name,
            'Metric':     'AUCNPR',
            'Year A':     yr_A,
            'Year B':     yr_B,
            'AUCNPR (A)': round(obs_A, 3),
            'AUCNPR (B)': round(obs_B, 3),
            '|Δ|':        round(obs_diff, 3),
            'p-value':    round(p_value, 3),
            'Significant (p<0.05)': p_value < 0.05,
        })

perm_npr_df_vl = pd.DataFrame(perm_npr_rows_vl)
perm_npr_df_vl.to_csv('biomarker_counts_fixed/Year Sensitivity - Vitals Labs - Permutation AUCNPR.csv', index=False)

n_sig = int(perm_npr_df_vl['Significant (p<0.05)'].sum())
print(f"AUCNPR: {n_sig} / {len(perm_npr_df_vl)} comparisons significant (p < 0.05)")
perm_npr_df_vl

AUCNPR: 0 / 33 comparisons significant (p < 0.05)


,DAG,Metric,Year A,Year B,AUCNPR (A),AUCNPR (B),|Δ|,p-value,Significant (p<0.05)
0,Control,AUCNPR,2020,2021,0.777,0.786,0.009,0.843,False
1,Control,AUCNPR,2020,2022,0.777,0.791,0.014,0.759,False
2,Control,AUCNPR,2021,2022,0.786,0.791,0.005,0.920,False
3,Correlation,AUCNPR,2020,2021,0.730,0.743,0.013,0.788,False
4,Correlation,AUCNPR,2020,2022,0.730,0.770,0.040,0.399,False
5,Correlation,AUCNPR,2021,2022,0.743,0.770,0.027,0.582,False
6,Simplified Clinician Consensus $\cup$ Simplifi...,AUCNPR,2020,2021,0.760,0.762,0.002,0.964,False
7,Simplified Clinician Consensus $\cup$ Simplifi...,AUCNPR,2020,2022,0.760,0.770,0.010,0.846,False
8,Simplified Clinician Consensus $\cup$ Simplifi...,AUCNPR,2021,2022,0.762,0.770,0.008,0.856,False
9,Simplified Clinician Consensus $\cup$ Simplifi...,AUCNPR,2020,2021,0.744,0.781,0.037,0.426,False
